In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc PyGithub networkx qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

*[APIリファレンス](https://docs.quantum.ibm.com/api/functions/kipu-optimization)を参照してください*

> **Note:** * Qiskit Functionsは、IBM Quantum&reg; Premium Plan、Flex Plan、およびOn-Prem（IBM Quantum Platform API経由）Planのユーザーのみが利用できる試験的な機能です。プレビューリリースの状態にあり、変更される場合があります。

## 概要

Kipu QuantumのIskay Quantum Optimizerを使用すると、IBM&reg;量子コンピューターを利用して複雑な最適化問題に取り組むことができます。このソルバーはKipuの最先端の[bf-DCQOアルゴリズム](https://doi.org/10.48550/arXiv.2409.04477)を活用しており、目的関数のみを入力として受け取り、問題の解を自動的に提供します。最大156量子ビットを含む最適化問題を処理でき、IBMの量子デバイスの全量子ビットを活用することが可能です。OptimizerはClassical変数と量子ビット間の1対1マッピングを使用するため、最大156個のバイナリ変数を持つ最適化問題に取り組むことができます。

Optimizerは制約なしバイナリ最適化問題を解くことができます。一般的に使用されるQUBO（Quadratic Unconstrained Binary Optimization）定式化に加えて、高次（HUBO）最適化問題もサポートしています。このソルバーは非変分量子アルゴリズムを使用し、大部分の計算を量子デバイス上で実行します。

以下では、使用されているアルゴリズムの詳細と関数の使用方法に関する簡単なガイド、さらにさまざまな規模と複雑さの問題インスタンスに対するベンチマーク結果を説明します。

## 説明

Optimizerは最先端の量子最適化アルゴリズムのすぐに使える実装です。量子ハードウェア上で高度に圧縮された量子回路を実行することで最適化問題を解きます。この圧縮は、量子システムの基本的な時間発展にcounterdiabatic項を導入することで実現されています。アルゴリズムは複数回のハードウェア実行を行って最終的な解を取得し、後処理と組み合わせます。これらのステップはOptimizerのワークフローにシームレスに統合されており、自動的に実行されます。

### Quantum Optimizerはどのように機能しますか？

このセクションでは、実装されているbf-DCQOアルゴリズムの基本を説明します。アルゴリズムの入門として[Qiskit YouTubeチャンネル](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s)も参照できます。

このアルゴリズムは量子システムの時間発展に基づいており、時間とともに変換されていきます。問題の解は、発展の最後において量子システムの基底状態に符号化されています。[断熱定理](https://en.wikipedia.org/wiki/Adiabatic_theorem)によれば、システムが基底状態に留まるためにはこの発展がゆっくりである必要があります。この発展をデジタル化したものがデジタル量子断熱計算（DQA）および有名なQAOAアルゴリズムの基礎となっています。しかし、必要とされるゆっくりとした発展は、問題サイズが大きくなるにつれて回路深度が増加するため、実現が困難になります。counterdiabaticプロトコルを使用することで、短い発展時間の間に生じる望ましくない励起を抑制しながら基底状態に留まることができます。この短い発展時間をデジタル化することで、より短い深度とより少ないエンタングリングゲートを持つ量子回路が得られます。

bf-DCQOアルゴリズムの回路は、DQAと比べてエンタングリングゲートが通常10倍少なく、標準的なQAOA実装と比べて3〜4倍少ないです。ゲート数が少ないため、ハードウェア上での回路実行中に発生するエラーが少なくなります。そのため、このOptimizerはエラー抑制やエラー軽減といった手法を必要としません。将来のバージョンにこれらを実装することで、解の品質をさらに向上させることができます。

bf-DCQOアルゴリズムは反復を使用しますが、非変分型です。アルゴリズムの各反復後、状態の分布が測定されます。得られた分布は、いわゆるバイアス場の計算に使用されます。バイアス場を利用することで、次の反復を以前に見つかった解の近くのエネルギー状態から開始することができます。このようにして、アルゴリズムは各反復でより低いエネルギーの解へと移行していきます。通常、約10回の反復で解に収束するのに十分であり、変分アルゴリズムの約100回程度と比べて、全体として必要な反復回数がはるかに少なくなります。

Optimizerはbf-DCQOアルゴリズムと古典的な後処理を組み合わせています。状態の分布を測定した後、局所探索が行われます。局所探索では、測定された解のビットがランダムに反転されます。反転後、新しいビット列のエネルギーが評価されます。エネルギーが低い場合、そのビット列が新しい解として採用されます。局所探索は量子ビット数に対して線形にスケールするため、計算コストが低くなっています。後処理が局所的なビット反転を修正するため、ハードウェアの不完全さや読み出しエラーによってしばしば引き起こされるビット反転エラーを補償します。

### ワークフロー

Quantum Optimizerのワークフローの模式図を以下に示します。

![ワークフロー](../docs/images/guides/kipu-optimization/workflow.svg "Quantum Optimizerのワークフロー")

Quantum Optimizerを使用することで、量子ハードウェア上での最適化問題の解法を以下のように簡略化できます。
* 問題の目的関数を定式化する
* Qiskit FunctionsからOptimizerにアクセスする
* Optimizerを実行して結果を収集する

## ベンチマーク

以下のベンチマーク指標は、Optimizerが最大156量子ビットの問題を効果的に扱えることを示しており、さまざまな問題タイプにおけるOptimizerの精度とスケーラビリティの概要を示しています。実際のパフォーマンス指標は、変数の数、目的関数における項の密度や局所性、多項式の次数など、特定の問題の特性によって異なる場合があることに注意してください。

以下の表には近似比（AR）が含まれており、その指標は次のように定義されます：
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
ここで$C$は目的関数、$C_{\textrm{min}}$、$C_{\textrm{max}}$はそれぞれその最小値と最大値、$C^{*}$は見つかった最良の解のコストです。したがって、AR=100\%は問題の基底状態が得られたことを意味します。

| 例                | 量子ビット数 | 近似比 | 合計時間（秒） | ランタイム使用量（秒） | ショット数合計 | 反復回数 |
| ----------------- | :--------------: | :------: | :------------: | :---------------: | :-------------------: | :------------------: |
| 重みなしMaxCut |        28        |   100%   |      180       |        30         |          30k          |          5           |
| 重みなしMaxCut |        30        |   100%   |      180       |        30         |          30k          |          5           |
| 重みなしMaxCut |        32        |   100%   |      180       |        30         |          30k          |          5           |
| 重みなしMaxCut |        80        |   100%   |      480       |        60         |          90k          |          9           |
| 重みなしMaxCut |       100        |   100%   |      330       |        60         |          60k          |          6           |
| 重みなしMaxCut |       120        |   100%   |      370       |        60         |          60k          |          6           |
| HUBO 1            |       156        |   100%   |      600       |        70         |         100k          |          10          |
| HUBO 2            |       156        |   100%   |      600       |        70         |         100k          |          10          |

- 28、30、32量子ビットのMaxCutインスタンスはibm_sherbrookeで実行されました。80、100、120量子ビットのインスタンスはHeron r2プロセッサーで実行されました。
- HUBOインスタンスもHeron r2プロセッサーで実行されました。

すべてのベンチマークインスタンスはGitHubで入手可能です（[Kipuベンチマークインスタンス](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)を参照）。これらのインスタンスを実行する例は[例3：ベンチマークインスタンス](#example-3-benchmark-instances)に記載されています。

## はじめに

このドキュメントでは、Iskay Quantum Optimizerの使用手順を説明します。その過程で、カタログから関数を読み込む方法や、問題を有効な入力に変換する方法を簡単に示しながら、さまざまなオプションパラメーターを試す方法についても説明します。

より詳細な例については、チュートリアル[Kipu QuantumのIskay Quantum OptimizerでMarket Split問題を解く](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)を参照してください。そこでは、Iskay SolverをMarket Split問題に適用する全プロセスを詳しく説明しています。Market Split問題は、厳密な需要目標を満たすためにマーケットをバランスの取れた販売地域に分割しなければならない現実世界のリソース配分の課題を表しています。

[IBM Quantum Platformダッシュボード](http://quantum.cloud.ibm.com/)で見つかるAPIキーを使用して認証し、Qiskit Functionを以下のように選択してください：

In [ ]:
# ruff: noqa: F821

> **Note:** 以下のコードは、認証情報が保存されていることを前提としています。まだ保存していない場合は、[IBM Cloudアカウントを保存する](/guides/functions#install-qiskit-functions-catalog-client)の指示に従ってAPIキーで認証してください。

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    # For `token`, use the 44-character API_KEY you created
    # and saved from the IBM Quantum Platform Home dashboard
    token="YOUR_API_KEY",
)

# Access Function
optimizer = catalog.load("kipu-quantum/iskay-quantum-optimizer")

## カスタム設定の例
異なる設定でIskayを構成する方法を以下に示します：

In [ ]:
custom_options = {
    "shots": 15_000,  # Higher shot count for better statistics
    "num_iterations": 12,  # More iterations for solution refinement
    "preprocessing_level": 1,  # Light preprocessing for problem simplification
    "postprocessing_level": 2,  # Maximum postprocessing for solution quality
    "transpilation_level": 3,  # Use higher transpilation level to optimize circuit
    "seed_transpiler": 42,  # Fixed seed for reproducible results
    "job_tags": ["custom_config"],  # Custom tracking tags
}

**シード最適化**：`seed_transpiler`はデフォルトで`None`に設定されていることに注意してください。これにより、トランスパイラーの自動最適化プロセスが有効になります。`None`の場合、システムは複数のシードでトライアルを開始し、最良の回路深度を生成するシードを選択します。これにより、各トランスパイルレベルの`max_trials`パラメーターのフル機能を活用できます。

**トランスパイルレベルのパフォーマンス**：`transpilation_level`の値を高くして`max_trials`の数を増やすと、トランスパイル時間が不可避的に長くなりますが、最終的な回路が変化しない場合もあります。これは特定の回路の構造や複雑さに大きく依存します。ただし、一部の回路や問題では、10トライアル（レベル1）と50トライアル（レベル5）の差が大きくなることがあるため、これらのパラメーターを探索することが解を正常に見つけるための鍵となる可能性があります。

## 例1：シンプルなコスト関数

スピン定式化でのコスト関数を考えます：
$$
C(x_0, x_1, x_2, x_3, x_4) = 1 + 1.5x_0 + 2x_1 + 1.3x_2 + 2.5x_0x_3 + 3.5x_1x_4 + 4x_0x_1x_2
$$

ここで$(x_0, ..., x_4) \in {-1, 1}^5$です。

このシンプルなコスト関数の解は
$$
(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)
$$
であり、最小値は$C^{*} = -6$です。

### 1. 目的関数を作成する

まず、次のように目的関数の係数を含む辞書を作成します：

In [ ]:
objective_func = {
    "()": 1,
    "(0,)": 1.5,
    "(1,)": 2,
    "(2,)": 1.3,
    "(0, 3)": 2.5,
    "(1, 4)": 3.5,
    "(0, 1, 2)": 4,
}

### 2. Optimizerを実行する
Optimizerを実行して問題を解きます。$(x_0, ..., x_4) \in {-1, 1}^5$であるため、`problem_type=spin`を設定する必要があります。

In [ ]:
# Setup options to run the optimizer
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. 結果を取得する
最適化問題の解はOptimizerから直接提供されます。

In [ ]:
print(job.result())

これにより、次の形式の辞書が表示されます：

```
{'solution': {'0': -1, '1': -1, '2': -1, '3': 1, '4': 1},
 'solution_info': {'bitstring': '11100',
  'cost': -13.8,
  'seed_transpiler': 42,
  'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}},
 'prob_type': 'spin'}
```

辞書`solution`が結果ベクトル$(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)$を表示していることに注目してください。

## 例2：MaxCut

MaxCutや最大独立集合などの多くのグラフ問題はNP困難問題であり、量子アルゴリズムとハードウェアをテストするのに理想的な候補です。この例では、Quantum OptimizerでMaxCut問題を3正則グラフで解く方法を示します。

この例を実行するには、`qiskit-ibm-catalog`に加えて`networkx`パッケージをインストールする必要があります。インストールするには、次のコマンドを実行してください：

In [ ]:
# %pip install networkx numpy

### 1. 目的関数を作成する
まずランダムな3正則グラフを生成します。このグラフに対してMaxCut問題の目的関数を定義します。

In [ ]:
import networkx as nx

# Create a random 3-regular graph
G = nx.random_regular_graph(3, 10, seed=42)


# Create the objective function for MaxCut in Ising formulation
def graph_to_ising_maxcut(G):
    """
    Convert a NetworkX graph to an Ising Hamiltonian for the max-cut problem.
    Args:
        G (networkx.Graph): The input graph.
    Returns:
        dict: The objective function of the Ising model
    """
    # Initialize the linear and quadratic coefficients
    objective_func = {}
    # Populate the coefficients
    for i, j in G.edges:
        objective_func[f"({i}, {j})"] = 0.5
    return objective_func


objective_func = graph_to_ising_maxcut(G)

### 2. Optimizerを実行する
Optimizerを実行して問題を解きます。

In [ ]:
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. 結果を取得する
結果を取得し、解のビット列を元のグラフノードにマッピングします。

In [ ]:
print(job.result())

MaxCut問題の解は、結果オブジェクトの`solution`サブ辞書に直接含まれています。

In [ ]:
maxcut_solution = job.result()["solution"]

## 例3：ベンチマークインスタンス
ベンチマークインスタンスはGitHubで入手可能です：[Kipuベンチマークインスタンス](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)

インスタンスは`pygithub`ライブラリを使用して読み込むことができます。インストールするには、次のコマンドを実行してください：

In [ ]:
# %pip install pygithub

ベンチマークインスタンスのパスは次のとおりです：

**Maxcut：**
- `'maxcut/maxcut_regular_3_100_nodes_weighted.json'`
- `'maxcut/maxcut_regular_3_140_nodes_weighted.json'`
- `'maxcut/maxcut_regular_3_150_nodes_weighted.json'`
- `'maxcut/maxcut_regular_4_130_nodes_weighted.json'`

**HUBO：**
- `'HUBO/hubo1_marrakesh.json'`
- `'HUBO/hubo2_marrakesh.json'`

HUBOインスタンスのベンチマーク性能を再現するには、バックエンドに`ibm_marrakesh`を選択し、`options`サブ辞書で`direct_qubit_mapping`を`True`に設定してください。

以下の例では、150ノードのMaxCutインスタンスを実行します。

In [ ]:
from github import Github
import urllib
import json
import ast

repo = "Kipu-Quantum-GmbH/benchmark-instances"
path = "maxcut/maxcut_regular_3_150_nodes_weighted.json"
gh = Github()
repo = gh.get_repo(repo)
branch = "main"
file = repo.get_contents(urllib.parse.quote(path), ref=branch)

# load json file with benchmark problem
problem_json = json.loads(file.decoded_content)

# convert objective function to compatible format
objective_func = {
    key: ast.literal_eval(value) for key, value in problem_json.items()
}


# Setup configuration to run the optimizer
options = {
    "shots": 5_000,
    "num_iterations": 5,
    "use_session": True,
    "direct_qubit_mapping": False,
}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": "<BACKEND-NAME>",
    "options": options,
}

job = optimizer.run(**arguments)

result = job.result()

## ユースケース
最適化ソルバーの典型的なユースケースは組み合わせ最適化問題です。金融、製薬、物流など多くの産業からの問題を解くことができます。いくつかの例を以下に示します。
* ポートフォリオ最適化（QUBO）：[学術論文](https://doi.org/10.1103/PhysRevApplied.22.054037)および[ホワイトペーパー](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* タンパク質折り畳み（HUBO）：[学術論文](https://doi.org/10.1103/PhysRevApplied.20.014024)
* 物流スケジューリング（QUBO）：[学術論文](https://doi.org/10.1103/PhysRevApplied.22.064068)
* ネットワーク最適化：[ウェビナー](https://www.youtube.com/watch?v=w5SrCIK88No)
* Market split（QUBO）：[チュートリアル](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

特定のユースケースに取り組み、専用のマッピングを開発することに興味がある場合は、お手伝いすることができます。[問い合わせてください。](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)

## サポートを受ける
サポートについては、support@kipu-quantum.comまで連絡してください。

## 次のステップ
- [Kipu QuantumのQuantum Optimizerへのアクセスをリクエストする](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
- このQiskit Functionの[APIリファレンス](https://docs.quantum.ibm.com/api/functions/kipu-optimization)を参照してください。
- チュートリアル[Kipu QuantumのIskay Quantum OptimizerでMarket Split問題を解く](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)をお試しください。
- [Romero, S. V., et al. (2025).  Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477](https://arxiv.org/abs/2409.04477)を参照してください。
- [Cadavid, A. G., et al. (2024).  Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898](https://arxiv.org/abs/2405.13898)を参照してください。
- [Chandarana, P., et al. (2025).  Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663](https://arxiv.org/abs/2505.08663)を参照してください。

## 追加情報
Iskayは、私たちの会社名Kipu Quantumと同様に、ペルーの言葉です。私たちはドイツのスタートアップですが、これらの言葉は共同創業者の一人の出身国に由来しており、そこではQuipuが紀元前2000年に人類が開発した最初の計算機の一つでした。